# 7.2 分布内与分布外泛化：配套实验

G6（本章A2组）· 仅7.2分节稿，不代表全章实验。

[正文](A2组-7.2-正文.md) · [习题](A2组-7.2-习题.md) · [运行与来源说明](A2组-7.2-编写说明.md)

顺序运行全部单元。实验一直接按分布抽样检验ID/OOD；实验二至四用固定网格观察输入范围与模型次数的影响。最后将相同评价原则用于一族热方程，计算解析参考并按完整参数工况划分，没有训练温度模型。

ID/OOD比较的是数据生成分布，而不是两批样本是否相同。输出由助手实际执行，不代表本组成员或B/C组独立复现。

## 环境、输入与检查方式

| 项目 | 设置 |
| --- | --- |
| 参考函数 | 无噪声sin(x)，横坐标按弧度代入，函数值无量纲 |
| 抽样实验 | 种子7201；依次独立抽训练41点U[-1,1]、ID测试1000点U[-1,1]、OOD测试1000点U[2,3]；预先固定三次模型 |
| 网格实验 | 训练[-1,1]上41个等距点；区间内40个交错点；区间外[2,3]上101点；比较3、5、8次 |
| 热方程设计 | 种子7202；训练/验证/ID参数各自独立来自U[0.05,0.10]，OOD来自U[0.20,0.50]；每工况51×51网格 |
| 指标与检查 | RMSE和有限点集最大绝对误差；输出有限，RMSE不超过点集最大误差；不预设OOD必败 |
| 环境与预算 | Python 3、NumPy、pandas，CPU小规模实验；随机生成器default_rng（PCG64），不需要GPU |

复用的41/40/101网格与三次拟合来自课程起步代码。新增抽样实验与参数划分是本稿的教学扩展，完整来源见编写说明。重跑只更新本包的 `数据/`。

核对正文时，按表内显示位数舍入；核对参考解答中保留10位小数的值时可用1e-8绝对容差。这只是数值复现检查，不是模型可接受误差或严格误差界。保存输出不能代替读者自己填写运行前预测。

In [1]:
from pathlib import Path
import platform
import runpy

import numpy as np
import pandas as pd

root = Path.cwd()
result = runpy.run_path(str(root / '配套代码' / 'experiments.py'))
table = pd.DataFrame(result['rows'])
print('Python:', platform.python_version())
print('NumPy:', np.__version__, 'pandas:', pd.__version__)
print('Sample seed:', result['SAMPLE_SEED'], '; parameter seed:', result['HEAT_SEED'])

Python: 3.12.7
NumPy: 1.26.4 pandas: 2.2.3
Sample seed: 7201 ; parameter seed: 7202


## 7.2.2 实验一：按分布抽样，独立比较ID与OOD

先预测：训练和ID测试都来自U[-1,1]，OOD测试改成U[2,3]，同一三次多项式的两种测试误差会怎样？

脚本预先固定种子7201、次数3及41/1000/1000点数，分别独立抽取三批输入；答案始终为sin(x)。ID测试点未参与训练；本例没有待选择的设置，因此不另设验证集。不根据输出挑种子或调模型。

以下表格对应正文实验一，完整精度保存在 `数据/sampled_results.csv`。单次抽样结果不等于总体风险，也不是置信保证。

In [2]:
sampled_table = pd.DataFrame(result['sampled_rows'])
assert sampled_table['points'].tolist() == [41, 1000, 1000]
assert np.isfinite(sampled_table[['rmse', 'max_abs_error']]).all().all()
assert (sampled_table['rmse'] <= sampled_table['max_abs_error']).all()
np.testing.assert_array_equal(
    result['sampled_domains']['train'],
    np.random.default_rng(result['SAMPLE_SEED']).uniform(-1, 1, 41),
)
sampled_table

,role,distribution,points,rmse,max_abs_error
0,train,"U[-1,1]",41,0.000289,0.000697
1,id_test,"U[-1,1]",1000,0.000319,0.001169
2,ood_test,"U[2,3]",1000,0.739198,1.413618


## 7.2.2 实验二：固定网格上的区间比较

保留课程起步实验：41个等距训练点、40个区间内交错点、101个区间外点。先预测误差，再运行。对应漫画、正文第二张数值表和图4左侧。

这是重新用固定网格拟合的模型，不是实验一的随机训练模型；两者不能混用数字。inside/outside表示位置，并非validation/test职责。此处为确定性近似检查，不冒充独立随机抽样ID/OOD实验。

In [3]:
degree3 = table.loc[table['degree'] == 3].copy()
degree3['domain'] = degree3['domain'].map({
    'train': '训练点', 'inside': '区间内新点', 'outside': '区间外新点'
})
print('outside/inside RMSE =', result['metadata']['outside_to_inside_rmse_ratio_degree3'])
degree3[['domain', 'points', 'rmse', 'max_abs_error']]

outside/inside RMSE = 2330.566704463312


,domain,points,rmse,max_abs_error
0,训练点,41,0.000341,0.000780
1,区间内新点,40,0.000312,0.000477
2,区间外新点,101,0.727300,1.391997


## 7.2.2 实验三：固定网格上的次数比较

保持实验二的训练网格和两组评价点不变，比较事先指定的三次、五次和八次多项式。对应正文第三张数值表和图4右侧，不是实验一随机模型的次数比较。这里的次数是多项式最高幂次，不是训练轮数。

如果根据这张表挑选模型，结果便参与了选择；之后仍需另一份未参与挑选的数据评价最终方案。

In [4]:
comparison = table.loc[table['domain'] != 'train'].pivot(
    index='degree', columns='domain', values='rmse'
)
comparison[['inside', 'outside']]

domain,inside,outside
degree,,
3,3.120702e-04,0.727300
5,1.939328e-06,0.136527
8,7.051202e-09,0.014932


**结果解释。** 当前无噪声正弦函数、当前网格和区间上，次数从3升到5、8，外推误差减小。这不支持“域外必败”，也不支持“次数越高永远越好”。完整精度的9行结果见 `数据/experiment_results.csv`。

## 7.2.2 实验四：固定模型，只换评价区间

对应课后题的数值小问。保持实验二的固定网格三次模型不变，在[1,2]、[2,3]、[3,4]分别取101个等距点评价。先写预测，再运行。这里只改变评价区间，没有重新拟合，也不是实验一的随机训练模型。

输出另存 `数据/interval_results.csv`。可以探索其他区间，但不能把一次观察写成所有函数、所有模型的普遍规律。

In [5]:
evaluation_intervals = [(1, 2), (2, 3), (3, 4)]
model = result['models'][3]
interval_rows = []
for left, right in evaluation_intervals:
    points = np.linspace(left, right, 101)
    error = model(points) - np.sin(points)
    interval_rows.append({
        'interval': f'[{left}, {right}]',
        'points': len(points),
        'rmse': float(np.sqrt(np.mean(error**2))),
        'max_abs_error': float(np.max(np.abs(error))),
    })
interval_table = pd.DataFrame(interval_rows)
interval_table.to_csv(root / '数据' / 'interval_results.csv', index=False, encoding='utf-8-sig')
interval_table

,interval,points,rmse,max_abs_error
0,"[1, 2]",101,0.069330,0.171153
1,"[2, 3]",101,0.727300,1.391997
2,"[3, 4]",101,3.258762,5.312809


## 7.2.2 小检验：相同观测与两个可能答案

此处模拟把真实函数名字遮住的情形。函数一为sin(x)；函数二在x≤1时与它相同，在x>1时增加(x−1)³。两者在观测区间相同，却在x=3给出不同答案。

这不是说两个函数同时都是真实函数，而是说已知数据无法告诉预测者哪一种才是真的。相差8的两个答案，任何同一预测至少离其中一个4以上（含4）。下面只核对数字，证明见正文。

In [6]:
observed_x = np.linspace(-1, 1, 1001)
f1 = np.sin(observed_x)
f2 = np.sin(observed_x) + np.maximum(observed_x - 1, 0)**3
assert np.array_equal(f1, f2)
answers = np.array([np.sin(3), np.sin(3) + 8])
midpoint = answers.mean()
assert np.isclose(answers[1] - answers[0], 8)
print('Answers at x=3:', answers)
print('Midpoint prediction:', midpoint)
print('Two errors:', np.abs(answers - midpoint))

Answers at x=3: [0.14112001 8.14112001]
Midpoint prediction: 4.1411200080598665
Two errors: [4. 4.]


## 7.2.3 案例：热方程的参考解与整工况划分

### 从一个函数到一族解

函数近似实验改变x的生成分布；这里改变热扩散参数ν的生成分布。同一ν决定一整套位置、时间上的解，模型输入应包含x、t、ν。模型能补齐同一工况的新网格点，不代表能预测另一工况。

只计算理想化热方程解析参考，不训练温度模型。所有量无量纲，0≤x≤1、0≤t≤1；方程u_t=νu_xx，两端温度固定0，初始温度sin(πx)。解析解为：

u(x,t;ν) = exp(−π²νt) sin(πx)。

求导得u_t=−π²νu、u_xx=−π²u，代入满足方程；t=0时满足初值，x=0和x=1时满足边界。下表手选ν=0.05、0.10、0.20、0.50，在x=0.5、t=1展示参考温度，与图6左侧曲线对应。这些手选参考参数不是后面的训练/测试抽样列表，解析温度也不是模型误差。

In [7]:
nu_values = np.array([0.05, 0.10, 0.20, 0.50])
x, t = 0.5, 1.0
reference_u = np.exp(-np.pi**2 * nu_values * t) * np.sin(np.pi * x)
reference_table = pd.DataFrame({
    'nu': nu_values, 'x': x, 't': t, 'analytic_u': reference_u
})
reference_table.to_csv(root / '数据' / 'heat_reference.csv', index=False, encoding='utf-8-sig')
reference_table

,nu,x,t,analytic_u
0,0.05,0.5,1.0,0.610498
1,0.10,0.5,1.0,0.372708
2,0.20,0.5,1.0,0.138911
3,0.50,0.5,1.0,0.007192


### 按参数生成分布留出完整工况

`experiments.py`用固定种子7202依次独立抽取四组参数：训练3种、验证2种、ID测试2种分别来自U[0.05,0.10]；OOD测试2种来自U[0.20,0.50]。每个参数对应一整套解，不是把同一工况的网格点随机拆给不同集合。完整精度保存在 `数据/heat_split.csv`，图文显示四位小数。

各工况采用完全相同的51×51坐标网格，初值和边界条件不变。ID/OOD判断以工况的生成分布为对象，不把工况内2601个点看成2601个独立材料样本。样本量只用于演示，不足以支撑可靠泛化估计。

注意本次ID测试参数高于三个训练参数的最大值，仍然来自同一个U[0.05,0.10]。比较的是生成分布，不是有限训练样本的最小/最大值。下面计算每组工况的解析温度并核验形状、初值与边界，没有训练模型。

In [8]:
group_table = pd.DataFrame(result['heat_rows'])
all_nu = group_table['nu'].to_numpy()
assert len(np.unique(all_nu)) == len(all_nu)
position = np.linspace(0, 1, 51)
time = np.linspace(0, 1, 51)
xx, tt = np.meshgrid(position, time)
for nu in all_nu:
    field = np.exp(-np.pi**2 * nu * tt) * np.sin(np.pi * xx)
    assert field.shape == (51, 51)
    assert np.isfinite(field).all()
    np.testing.assert_allclose(field[:, [0, -1]], 0, atol=1e-15)
    np.testing.assert_allclose(field[0], np.sin(np.pi * position), atol=1e-15)
print('Parameter seed:', result['HEAT_SEED'])
print('9 distinct cases; 51 x 51 grid per case; no learned model.')
group_table

Parameter seed: 7202
9 distinct cases; 51 x 51 grid per case; no learned model.


,role,distribution,nu
0,train,"U[0.05,0.10]",0.061301
1,train,"U[0.05,0.10]",0.076160
2,train,"U[0.05,0.10]",0.078493
3,validation,"U[0.05,0.10]",0.071001
4,validation,"U[0.05,0.10]",0.088893
5,id_test,"U[0.05,0.10]",0.085271
6,id_test,"U[0.05,0.10]",0.099652
7,ood_test,"U[0.20,0.50]",0.375659
8,ood_test,"U[0.20,0.50]",0.467917


## 实验后的结论边界

- 实验一按明确的连续输入分布独立抽样，比较ID/OOD；一次固定种子输出不是总体误差或置信保证。
- 实验二至四保留确定性网格结果，不冒充随机同分布抽样；不能与实验一混用数字。
- 同范围内频率改变也可构成OOD，但正文的25%/75%情形仅作定义示意，此处没有运行其误差实验。
- 热方程只生成解析参考并按参数分布划分完整工况，没有训练模型，也没有它的泛化成绩。
- 工况内网格点不能充当独立材料样本；小规模参数组只演示方法，不足以估计可靠泛化表现。
- 全部7张图由 `配套代码/generate_figures.py` 生成，绘图读取参数分组CSV，不改写实验数据。

请用两句话写出：ID/OOD究竟比较什么？本节实际检验了什么、还没检验什么？